# 믹서 MCP에서 도구 목록을 받아 봐요

이 노트북은 강사가 만든 **실제 MCP 서버 3.3.0**을 Colab에서 실행해요. AI 모델과 실제 믹서는 사용하지 않아요.

1. 첫 번째 코드 칸에서 필요한 프로그램을 설치해요.
2. 두 번째 코드 칸에서 도구 24개의 목록을 받아요.
3. 세 번째 코드 칸에서 잘못된 채널 `40`을 보내고, 거부 이유를 확인해요.

각 코드 칸 왼쪽의 ▶를 누르세요. 설치에는 인터넷이 필요해요. 처음 실행할 때는 시간이 걸릴 수 있어요.

장비 주소를 입력하거나 `connection_connect`를 호출하지 않아요. **채널 40은 그대로 두세요.** 오늘은 올바른 볼륨 변경이 아니라 입력 검사를 확인해요.


In [ ]:
# 이 칸은 MCP와 Node.js를 준비해요. 내 컴퓨터가 아니라 Colab에 설치돼요.
import importlib.metadata
import os
import shutil
import subprocess
import sys
import sysconfig

packages = {"mcp": "1.26.0", "nodejs-wheel": "22.14.0"}
missing = []
for name, version in packages.items():
    try:
        installed = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        missing.append(f"{name}=={version}")
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)

# 방금 설치한 실행 파일을 먼저 찾게 해요.
os.environ["PATH"] = sysconfig.get_path("scripts") + os.pathsep + os.environ["PATH"]
node_version = subprocess.check_output(["node", "--version"], text=True).strip()
assert node_version == "v22.14.0", node_version
assert shutil.which("npx"), "실행 파일을 찾지 못했어요. 설치 오류를 강사에게 보여 주세요."
print("준비됐어요. 다음 코드 칸을 실행하세요.")

## 도구 24개를 받아요

다음 칸을 실행하면 도구 수와 `channel_set_volume`의 명세가 나와요. `name`, `description`, `inputSchema`를 찾아보세요.

설치나 다운로드 중에 멈추면 오류 메시지를 그대로 강사에게 보여 주세요. 결과가 없는데 성공한 것으로 적지 않아요.

In [ ]:
# import는 다른 파일이나 패키지에 있는 이름을 여기서 쓰겠다는 뜻이에요.
import asyncio
import json
import os
from datetime import timedelta
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from mcp.shared.exceptions import McpError

# 이 서버 프로그램은 Colab 안에서 실행돼요. 장비 주소는 넘기지 않아요.
server = StdioServerParameters(
    command=shutil.which("npx"),
    args=["-y", "x-m32-mcp-server@3.3.0"],
    env={"PATH": os.environ["PATH"], "HOME": os.environ["HOME"]},
)

async def read_tools():
    async with stdio_client(server) as (reader, writer):
        async with ClientSession(reader, writer, read_timeout_seconds=timedelta(seconds=45)) as session:
            await session.initialize()  # 먼저 서로 사용할 규격을 확인해요.
            response = await session.list_tools()  # tools/list 요청이에요.
            return response.tools

# await는 서버에서 결과가 올 때까지 기다려요.
tools = await asyncio.wait_for(read_tools(), timeout=120)
assert len(tools) == 24, f"예상한 도구는 24개인데 {len(tools)}개가 왔어요. 강사에게 알려 주세요."
print(f"도구 {len(tools)}개를 받았어요.")
volume = next(tool for tool in tools if tool.name == "channel_set_volume")
print(json.dumps({"name": volume.name, "description": volume.description,
                  "inputSchema": volume.inputSchema}, ensure_ascii=False, indent=2))

## 채널 40을 요청하면 어떻게 될까요?

명세에서 채널의 최댓값을 먼저 확인하세요. 다음 칸을 실행하면 실제 서버의 거부 메시지가 나와요.

채널 번호를 정상 범위로 바꾸지 않아요. 이 실습에서는 믹서에 연결하지 않고, 허용 범위를 벗어난 입력만 보내요.

In [ ]:
async def check_invalid_channel():
    async with stdio_client(server) as (reader, writer):
        async with ClientSession(reader, writer, read_timeout_seconds=timedelta(seconds=45)) as session:
            await session.initialize()
            response = await session.list_tools()
            tool = next(t for t in response.tools if t.name == "channel_set_volume")
            # 규격이 바뀌면 요청하지 않고 멈춰요.
            assert tool.inputSchema["properties"]["channel"]["maximum"] == 32
            try:
                result = await session.call_tool(
                    "channel_set_volume", {"channel": 40, "value": -20, "unit": "db"}
                )
            except McpError as error:
                # MCP 프로토콜 오류로 돌아오는 경우예요.
                assert error.error.code == -32602, str(error)
                message = str(error)
            else:
                # 도구 실행 결과에 오류가 표시되는 경우예요.
                assert result.isError, "거부되지 않았어요. 강사에게 알려 주세요."
                message = "\n".join(c.text for c in result.content if c.type == "text")
            assert "32" in message and "channel" in message, message
            return message

message = await asyncio.wait_for(check_invalid_channel(), timeout=120)
print(message)
print("확인했어요: 채널 40은 입력 검사에서 거부됐어요.")

## 확인한 내용을 적어요

- 받은 도구 수:
- 볼륨 조절 Tool이 허용하는 채널 범위:
- 채널 40을 거부한 이유:

세 항목을 적으면 끝이에요. MCP 서버는 기능을 제공하는 프로그램이고, 입력을 검사한 뒤 실제 기능으로 연결한다는 것을 확인했어요.

이 결과는 실제 믹서가 움직였다는 증거는 아니에요.

출처: [XM32-MCP](https://github.com/GoBeromsu/XM32-MCP), [MCP Tools](https://modelcontextprotocol.io/specification/2025-06-18/server/tools).